# Быстрое переобучение адаптера под версию промпта (Colab A100 80 ГБ)

Один проход: **эпизоды из Drive** + **отправка коллеги** (его `prompts/system.md`, `eval_config.yaml`) ->
обучающий набор с его промптом -> адаптер LoRA -> готовый `submission.zip` в Drive. По желанию -- замер на 20 задачах.

Два режима (`MODE` в ячейке 0):
* `reward` -- обучение с наградой (`train.pg_lora`): по повторам одной задачи ходы удачных попыток становятся
  вероятнее, неудачных -- менее; штрафы за отсутствие правки и длинные рассуждения учат думать коротко и действовать;
* `sft` -- только решённые эпизоды (`train.build_sft_data` + `train.manual_lora_qat`).

Зачем: промпт коллеги меняется часто, а адаптер лучше всего работает с тем промптом, под который его учили.
Эпизоды переиспользуются: системный промпт подставляется заново при сборке набора, новый сбор не нужен.

Порядок: **0** параметры (единственная ячейка, которую надо править), **1** карта, **2** код, **3** Drive,
**4** модель, **5** набор, **6** обучение (vLLM НЕ запускать), **7** сохранение, **8** архив отправки,
**9-11** замер (необязательно). После перезапуска среды выполните 0-4 и продолжайте с нужной ячейки.

## 0. Параметры
Всё лежит в Drive в папке `gemma_adapters`:
* `SUBMISSION_ZIP` -- отправка коллеги (zip с `agent.yaml`, `prompts/system.md`, `eval_config.yaml`, ...);
* `RUNS` -- zip-архивы или папки с эпизодами (`results.jsonl` + `messages/`), например `mimo_p1_full_150.zip`;
* 20 оценочных задач в обучение не попадают никогда.

In [ ]:
MODE = "reward"                                # "reward" -- обучение с наградой по повторам; "sft" -- только решённые
ADAPTER = "pg_v13_a"                           # имя нового адаптера (латиница, цифры, _)
SUBMISSION_ZIP = "submissions/teammate_v13.zip"  # путь внутри gemma_adapters
RUNS = ["traj_think.zip", "mimo_p1_full_150.zip", "rl_mimo14_r8", "rl_comp19_r8"]
TURNS = 6                 # ходов на эпизод в обучении
MAX_LEN = 8192            # потолок на A100 80 ГБ; при нехватке памяти 6144
# reward: награда = 1 за пройденные тесты - штраф без правки - штраф за долю длинных рассуждений
NO_EDIT_PENALTY = 0.2
LONG_THOUGHT_CHARS = 2400  # ~600 токенов
LENGTH_PENALTY = 0.2
KL_COEF = 0.05
LR_REWARD = 5e-5          # раунд 1 с 2e-5 почти не сдвинул модель
EPOCHS_REWARD = 2
# sft
MAX_THOUGHT_CHARS = 1500  # ходы с более длинным рассуждением не берём: учим думать коротко
PER_TASK = 3              # не больше стольких решённых эпизодов на задачу
HOLDOUT_FRAC = 0.15       # доля решённых задач, отложенных для проверки
EPOCHS = 2

EVAL_IDS = """
fastapi_14303 fastapi_14786 fastapi_14361 requests_7315 rich_2725 rich_3882 fastapi_12942 fastapi_14356
fastapi_14246 fastapi_14258 fastapi_14306 fastapi_14371 fastapi_14479 fastapi_14487 fastapi_14512 fastapi_14605
rich_3052 rich_3180 rich_3486 rich_3930
""".split()
assert len(set(EVAL_IDS)) == 20

## 1. Карта (нужна A100 80 ГБ)

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip()
print(out)
name, total, used = [x.strip() for x in out.split(",")]
assert int(total) >= 79000, f"нужна карта на 80 ГБ, а здесь {total} МиБ: смените среду выполнения"
assert int(used) < 2000, f"занято {used} МиБ: остановите vLLM (!pkill -f vllm) или перезапустите среду"

## 2. Код и зависимости

In [ ]:
import os
if not os.path.exists("/content/gemma4-swe-rl"):
    !git clone -q https://github.com/aletonfdg/gemma4-swe-rl.git /content/gemma4-swe-rl
%cd /content/gemma4-swe-rl
!git pull -q
!git log --oneline | head -3
# torch и transformers в Colab уже есть; не обновляем их, чтобы не сломать совместимость
!pip install -q compressed-tensors accelerate safetensors huggingface_hub kagglehub openai pyyaml requests
import torch, transformers
print("torch", torch.__version__, "transformers", transformers.__version__)

## 3. Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/gemma_adapters"
os.makedirs(DRIVE, exist_ok=True)
assert os.path.exists(f"{DRIVE}/{SUBMISSION_ZIP}"), f"нет {DRIVE}/{SUBMISSION_ZIP}"
missing = [r for r in RUNS if not os.path.exists(f"{DRIVE}/{r}")]
print("нет в Drive (будут пропущены):", missing)
RUNS = [r for r in RUNS if r not in missing]
assert RUNS, "нет ни одной папки с эпизодами"

## 4. Базовая модель в `/content/model` (если уже есть, ничего не качает)

In [ ]:
import getpass
MODEL_DIR = "/content/model"
HF_REPO = "google/gemma-4-31b-it-qat-w4a16-ct"
KAGGLE_MODEL = "google/gemma-4/transformers/gemma-4-31b-it-qat-w4a16-ct"
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    try:
        from huggingface_hub import snapshot_download
        hf_token = getpass.getpass("HF токен (Enter, если не нужен): ").strip() or None
        snapshot_download(HF_REPO, local_dir=MODEL_DIR, token=hf_token)
    except Exception as e:
        print("HF не получилось:", type(e).__name__, str(e)[:300])
        import kagglehub
        os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("KAGGLE_API_TOKEN (KGAT_...): ").strip()
        path = kagglehub.model_download(KAGGLE_MODEL)
        !cp -r "{path}"/. {MODEL_DIR}/
os.environ["GEMMA_MODEL_DIR"] = MODEL_DIR
!du -sh {MODEL_DIR}

## 5. Обучающий набор с промптом из отправки
Распаковка отправки и эпизодов, затем `train.build_sft_data` с её `prompts/system.md`. Смотрите на строки
«решённых задач с годными траекториями» и «записано примеров»: меньше ~100 примеров -- обучение почти ничего не даст.

In [ ]:
import json, shutil, zipfile
SUB_DIR = "/content/submission_src"
shutil.rmtree(SUB_DIR, ignore_errors=True)
zipfile.ZipFile(f"{DRIVE}/{SUBMISSION_ZIP}").extractall(SUB_DIR)
# отправку иногда пакуют с папкой верхнего уровня: ищем agent.yaml
root = next(d for d, _, files in os.walk(SUB_DIR) if "agent.yaml" in files)
if root != SUB_DIR:
    tmp = SUB_DIR + "_tmp"; shutil.move(root, tmp); shutil.rmtree(SUB_DIR); shutil.move(tmp, SUB_DIR)
SYSTEM = f"{SUB_DIR}/prompts/system.md"
assert os.path.exists(SYSTEM), "в отправке нет prompts/system.md"
print(open(f"{SUB_DIR}/agent.yaml").read())
print(open(f"{SUB_DIR}/eval_config.yaml").read() if os.path.exists(f"{SUB_DIR}/eval_config.yaml") else "eval_config.yaml нет")

run_dirs = []
for r in RUNS:
    src = f"{DRIVE}/{r}"
    dst = f"/content/runs/{os.path.splitext(os.path.basename(r))[0]}"
    if not os.path.exists(dst):
        if r.endswith(".zip"):
            zipfile.ZipFile(src).extractall(dst)
        else:
            shutil.copytree(src, dst)
    # архивы бывают с вложенными папками: берём каждую папку, где есть results.jsonl
    run_dirs += [d for d, _, files in os.walk(dst) if "results.jsonl" in files]
print("папок с эпизодами:", len(run_dirs), *run_dirs, sep="\n  ")

json.dump(EVAL_IDS, open("/content/eval_ids.json", "w"))
if MODE == "reward":
    PG_ARGS = ["--system", SYSTEM, "--exclude-from /content/eval_ids.json", "--turns", str(TURNS),
               "--no-edit-penalty", str(NO_EDIT_PENALTY), "--long-thought-chars", str(LONG_THOUGHT_CHARS),
               "--length-penalty", str(LENGTH_PENALTY), "--runs"] + run_dirs
    # только статистика групп: сколько задач со смешанным исходом и ходов в плюс/минус
    dry_cmd = " ".join(["python -m train.pg_lora --dry-run"] + PG_ARGS)
    !{dry_cmd}
else:
    DATA = f"data/{ADAPTER}.jsonl"
    os.makedirs("data", exist_ok=True)
    cmd = ["python -m train.build_sft_data --system", SYSTEM, "--out", DATA, "--turns", str(TURNS),
           "--max-thought-chars", str(MAX_THOUGHT_CHARS), "--per-task", str(PER_TASK),
           "--holdout-frac", str(HOLDOUT_FRAC), "--exclude-from /content/eval_ids.json --runs"] + run_dirs
    build_cmd = " ".join(cmd)
    !{build_cmd}
    n = sum(1 for l in open(DATA, encoding="utf-8") if l.strip())
    print("примеров:", n)
    assert n > 0, "набор пуст"
    shutil.copy(DATA, f"{DRIVE}/{ADAPTER}.jsonl"); shutil.copy(DATA + ".tasks.json", f"{DRIVE}/{ADAPTER}.tasks.json")

## 6. Обучение (vLLM не запускать)
Отдельный процесс: после него память освобождается сама. Лог: `/content/{ADAPTER}_train.log`.
При `CUDA out of memory` поставьте в ячейке 0 `MAX_LEN = 6144` и повторите ячейки 0 и 6.

In [ ]:
LOG = f"/content/{ADAPTER}_train.log"
if MODE == "reward":
    # в логе: "разность logp удачных и неудачных" на обучающих ходах должна вырасти (иначе модель не учится),
    # на отложенных задачах -- тоже (иначе не обобщает); kl -- малой (< ~0.05)
    train_cmd = " ".join(["python -m train.pg_lora --name", ADAPTER, "--decompress --max-len", str(MAX_LEN),
                          "--kl-coef", str(KL_COEF), "--lr", str(LR_REWARD), "--epochs", str(EPOCHS_REWARD)]
                         + PG_ARGS)
else:
    train_cmd = (f"python -m train.manual_lora_qat train --data {DATA} --name {ADAPTER} --decompress "
                 f"--epochs {EPOCHS} --grad-accum 2 --max-len {MAX_LEN}")
!{train_cmd} 2>&1 | tee {LOG}

## 7. Сохранить адаптер и лог в Drive

In [ ]:
assert os.path.exists(f"adapters/{ADAPTER}/adapter_model.safetensors"), "адаптер не сохранился, смотрите лог"
!cp -r adapters/{ADAPTER} {DRIVE}/ && cp {LOG} {DRIVE}/
!ls -la {DRIVE}/{ADAPTER}
!grep -iE "eval|loss" {LOG} | tail -8

## 8. Архив отправки: отправка коллеги + строка `adapter:` + файлы адаптера
Проверки по правилам: файлы в корне архива, распакованный размер < 3 ГиБ. Результат: `gemma_adapters/upload_{ADAPTER}/submission.zip`.

In [ ]:
import re
OUT_DIR = f"/content/upload_{ADAPTER}"
shutil.rmtree(OUT_DIR, ignore_errors=True)
shutil.copytree(SUB_DIR, OUT_DIR)
shutil.rmtree(f"{OUT_DIR}/adapters", ignore_errors=True)   # чужие адаптеры из отправки не берём
os.makedirs(f"{OUT_DIR}/adapters/{ADAPTER}")
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(f"adapters/{ADAPTER}/{f}", f"{OUT_DIR}/adapters/{ADAPTER}/{f}")
agent = open(f"{OUT_DIR}/agent.yaml", encoding="utf-8").read()
agent = re.sub(r"(?m)^adapter:.*\n?", "", agent).rstrip("\n") + f"\nadapter: {ADAPTER}\n"
open(f"{OUT_DIR}/agent.yaml", "w", encoding="utf-8").write(agent)
print(agent)

size = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(OUT_DIR) for f in fs)
print(f"распакованный размер: {size / 2**30:.3f} ГиБ")
assert size < 3 * 2**30, "больше 3 ГиБ"
ZIP = f"/content/submission_{ADAPTER}.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for d, _, fs in os.walk(OUT_DIR):
        for f in fs:
            p = os.path.join(d, f)
            z.write(p, os.path.relpath(p, OUT_DIR))
print(*sorted(zipfile.ZipFile(ZIP).namelist()), sep="\n")
os.makedirs(f"{DRIVE}/upload_{ADAPTER}", exist_ok=True)
shutil.copy(ZIP, f"{DRIVE}/upload_{ADAPTER}/submission.zip")
print("готово:", f"{DRIVE}/upload_{ADAPTER}/submission.zip")

## 9. (Замер) vLLM с новым адаптером
Дальше необязательно: прогон 20 оценочных задач с тем же промптом и бюджетом, что в отправке. Около часа.

In [ ]:
import time, requests
!pip install -q vllm
cmd = ["vllm", "serve", MODEL_DIR, "--served-model-name", "gemma-4-31b-it-qat-w4a16-ct",
       "--quantization", "compressed-tensors", "--dtype", "bfloat16",
       "--max-model-len", "32768", "--gpu-memory-utilization", "0.92", "--max-num-seqs", "4",
       "--enable-lora", "--max-lora-rank", "64", "--max-loras", "1",
       "--lora-modules", f"{ADAPTER}=/content/gemma4-swe-rl/adapters/{ADAPTER}",
       "--enable-auto-tool-choice", "--tool-call-parser", "gemma4", "--reasoning-parser", "gemma4",
       "--default-chat-template-kwargs", '{"enable_thinking": true}', "--port", "8000"]
proc = subprocess.Popen(cmd, stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
t0 = time.time()
while time.time() - t0 < 1500 and proc.poll() is None:
    try:
        if requests.get("http://127.0.0.1:8000/health", timeout=3).status_code == 200:
            print("vLLM готов за %.0f c" % (time.time() - t0)); break
    except requests.RequestException:
        pass
    time.sleep(5)
if proc.poll() is not None:
    print("vLLM упал:"); print(open("/content/vllm.log").read()[-3000:])
print([m["id"] for m in requests.get("http://127.0.0.1:8000/v1/models").json()["data"]])

## 10. (Замер) Сервер песочницы на ноутбуке
Docker Desktop запущен, ноутбук не засыпает. Два окна PowerShell из папки проекта:
```
python -m sim.mimo_server --comp data/comp_full --valid data/benchmark_valid.json --max-sessions 4 --ttl 2400
cloudflared tunnel --url http://127.0.0.1:8765
```

In [ ]:
import getpass, urllib.request
os.environ["SWE_SANDBOX_URL"] = input("Адрес туннеля: ").strip().rstrip("/")
os.environ["SWE_SANDBOX_TOKEN"] = getpass.getpass("Токен сервера: ").strip()
req = urllib.request.Request(os.environ["SWE_SANDBOX_URL"] + "/tasks",
                             headers={"Authorization": "Bearer " + os.environ["SWE_SANDBOX_TOKEN"]})
server_ids = set(json.load(urllib.request.urlopen(req, timeout=30))["result"])
missing = [i for i in EVAL_IDS if i not in server_ids]
assert not missing, f"сервер не знает {missing}: запущен ли он с --comp data/comp_full?"
print("сервер готов, задач:", len(server_ids))

## 11. (Замер) 20 задач с промптом и бюджетом отправки
Температура 0.7, как в прежних замерах (`sft_think_v1`, `pref_edit_v1`: 7 из 20). Шум около ±3 задач:
разница меньше 3-4 задач ничего не значит.

In [ ]:
import yaml
ev = (yaml.safe_load(open(f"{SUB_DIR}/eval_config.yaml")) or {}).get("evaluation", {}) if os.path.exists(f"{SUB_DIR}/eval_config.yaml") else {}
budget = []
if ev.get("max_tool_calls"):
    budget += ["--tool-calls", str(ev["max_tool_calls"])]
if ev.get("max_time_minutes"):
    budget += ["--time-minutes", str(ev["max_time_minutes"])]
EVAL_OUT = f"results/{ADAPTER}_eval20"
!rm -rf {EVAL_OUT}
run_cmd = " ".join(["python -m sim.run_mimo_baseline --model", ADAPTER, "--temperature 0.7 --repeats 1 --workers 4",
                    "--thinking on --no-analyzer --clean-start --system-file", SYSTEM, *budget,
                    "--out", EVAL_OUT, "--ids", *EVAL_IDS])
print(run_cmd)
!{run_cmd}
last = {}
for l in open(f"{EVAL_OUT}/results.jsonl", encoding="utf-8"):
    r = json.loads(l); last[(r["instance_id"], r.get("rep", 0))] = r
print("решено:", sum(r["passed"] for r in last.values()), "из", len(last))
!zip -qr {DRIVE}/{ADAPTER}_eval20.zip {EVAL_OUT} && echo "копия в Drive"

## Приложение
Остановить vLLM: `!pkill -f vllm` (если память не освободилась, перезапустите среду).